In [ ]:
import re, os
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class Spec:
    snippet_id: str
    summary: str
    requirements: List[str]
    edge_cases: List[str]
    signature_hint: Dict[str, Any]
    constraints: List[str]
    tests_hint: List[Dict[str, Any]]

PROMPT = """You are a precise code generator.
    Implement function "{name}" with signature args={args}.
    Requirements (must all hold):
    {reqs}
    Constraints: {constraints}
    Return clean, executable Python code with only the function and minimal helpers.
    Do not import external libraries unless required for correctness."""

def build_prompt(spec: Spec):
    reqs = "\n".join([f"- {r}" for r in spec.requirements])
    cons = ", ".join(spec.constraints)
    return PROMPT.format(name=spec.signature_hint["name"], args=spec.signature_hint["args"], reqs=reqs, constraints=cons)

In [ ]:
import openai

def call_llm(model: str, prompt: str, temperature: float=0.0):
    # determine provider from model string
    provider = model.split(":")[0]

    # strip provider prefix
    model = model.split(":")[1]

    # call appropriate LLM provider
    if (provider == "openai"):        
        # openai api key
        openai.api_key = os.getenv("OPENAI_API_KEY")

        # check if api key exists
        if not openai.api_key:
            raise RuntimeError("Please set the OPENAI_API_KEY environment variable.")
        
        # call OpenAI
        response = openai.chat.completions.create(
            model=model,  
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=temperature
        )

        # return the output text
        return response.choices[0].message.content
    else:
        raise ValueError(f"Unknown provider: {provider}")

def extract_code(text: str):
    # regex to extract code block
    m = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
    # if no code block, return the whole text stripped
    return m.group(1).strip() if m else text.strip()

In [ ]:
import json, time

def gen_candidates(spec: Spec, models: List[str], temps=[0.0,0.7], k=2):
    # make candidates and cache dirs
    os.makedirs("candidates", exist_ok=True)
    os.makedirs("cache", exist_ok=True)

    # cache path
    cache_path = f"cache/{spec.snippet_id}.jsonl"

    # seen contains keys of already used model,temp,k combos
    seen = set()

    # out contains list of output file paths
    outs = []

    # loop over models, temps, k
    for m in models:
        for t in temps:
            for i in range(k):
                # construct unique key of model,temp,k combo
                key = f"{m}:{t}:{i}"

                # if already seen, skip
                if key in seen: continue

                # call llm and get output code
                raw = call_llm(m, build_prompt(spec), temperature=t)
                code = extract_code(raw or "")

                # consruct unique filename
                fname = f"candidates/{spec.snippet_id}__{m}__t{t}__k{i}.py"

                # save code and info
                with open(fname, "w") as f: f.write(code)
                with open(cache_path, "a") as f: f.write(json.dumps({"key":key,"path":fname})+"\n")

                # add to outs and seen
                outs.append(fname); seen.add(key)

                # sleep to avoid rate limit
                time.sleep(0.2)
    return outs


In [ ]:
# --- Test cell for Notebook 02 ---

import os, json, ast

# 1) Load a spec produced by Notebook 01

# find specs folder
spec_files = sorted([f for f in os.listdir("specs") if f.endswith(".json")])
assert spec_files, "No spec JSON found in ./specs. Run Notebook 01 first."

# load the first spec
spec_path = os.path.join("specs", spec_files[0])
with open(spec_path, "r") as f:
    spec_data = json.load(f)
spec = Spec(**spec_data)

print(f"Loaded spec: {spec_path}")
print("Function:", spec.signature_hint.get("name"), "Args:", spec.signature_hint.get("args"))

# 2) Build prompt
prompt = build_prompt(spec)
print(f"Prompt: {prompt}")

# 3) Configure llm models
models = [
    "openai:gpt-4o-mini"
    # e.g., "openai:gpt-4o-mini", "anthropic:claude-3-5-sonnet", "vertex:codegemini"
]
assert models, "Please populate the `models` list with at least one real model ID."

# set temps and k
temps = [0.0, 0.7]   # deterministic + diverse
k = 2                # candidates per (model, temp)

# 4) Generate candidates
cand_paths = gen_candidates(spec, models=models, temps=temps, k=k)
print(f"\n[✓] Generated {len(cand_paths)} candidates.")

# 5) Validate candidates: ensure code blocks extracted and Python-parsable
def is_parsable_python(path: str) -> bool:
    try:
        code = open(path, "r", encoding="utf-8").read()
        ast.parse(code)
        return True
    except Exception:
        return False

valid, invalid = [], []
for p in cand_paths:
    (valid if is_parsable_python(p) else invalid).append(p)

print(f"[Validation] Parsable: {len(valid)} | Non-parsable: {len(invalid)}")
if invalid:
    print("  -> Non-parsable files:")
    for p in invalid:
        print("     -", p)


Loaded spec: specs/e2ed22abfca3.json
Function: factorial Args: ['n']
Prompt: You are a precise code generator.
    Implement function "factorial" with signature args=['n'].
    Requirements (must all hold):
    - Produce deterministic output for identical inputs.
- Handle empty or None-like inputs gracefully where applicable.
- Raise ValueError on clearly invalid inputs.
    Constraints: No network calls, No filesystem writes, Deterministic behavior
    Return clean, executable Python code with only the function and minimal helpers.
    Do not import external libraries unless required for correctness.

[✓] Generated 4 candidates.
[Validation] Parsable: 4 | Non-parsable: 0

[Sample candidate: candidates/e2ed22abfca3__openai:gpt-4o-mini__t0.0__k0.py]
def factorial(n):
    if n is None or (isinstance(n, int) and n < 0):
        raise ValueError("Input must be a non-negative integer.")
    
    if n == 0:
        return 1
    
    if not isinstance(n, int):
        raise ValueError("Input 